# Assignment 9: Advanced Flows — Continuous Normalising Flows and Flow Matching

This notebook covers **Continuous Normalising Flows (CNFs)** and **Flow Matching** — moving from discrete invertible layers to continuous-time generative models defined by velocity fields.

Topics covered (Lecture 9):
- From discrete invertible layers to continuous flows defined by velocity fields
- ODE-based generation: solve dx/dt = v_θ(x, t) from t=0 to t=1
- The continuity equation and how probability density evolves under a flow
- Divergence as a measure of local expansion or compression
- Flow matching: a simpler training objective without solving ODEs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)

---
## Part 1: Velocity Fields and Trajectories

A **continuous normalising flow** is defined by a velocity field $v: [0,1] \times \mathbb{R}^D \to \mathbb{R}^D$.
Points evolve as:
$$\frac{dx_t}{dt} = v(x_t, t), \quad x_0 \sim p_0 \text{ (base distribution)}$$

Generation: solve the ODE from $t=0$ to $t=1$ to map noise samples into data space.

Unlike discrete flows, the velocity field $v_\theta$ can be an **unconstrained** neural network — invertibility is guaranteed by the existence and uniqueness of ODE solutions.

In [ ]:
def constant_velocity(x, t, a):
    """Constant velocity field: v(x, t) = a (same everywhere).

    Args:
        x : np.ndarray (D,) or (N, D)
        t : float, time
        a : np.ndarray (D,) or float, velocity

    Returns:
        v : same shape as x
    """
    # TODO: return np.broadcast_to(np.asarray(a), x.shape).copy()
    return np.broadcast_to(np.asarray(a), x.shape).copy()


def rotating_velocity_2d(x, t):
    """Clockwise rotating velocity field: v(x, t) = [-x2, x1].

    This field produces circular orbits around the origin (angular velocity = 1 rad/s).

    Args:
        x : np.ndarray (N, 2)
        t : float

    Returns:
        v : np.ndarray (N, 2)
    """
    # TODO:
    # return np.stack([-x[:, 1], x[:, 0]], axis=1)
    # This produces circular orbits around the origin
    return np.stack([-x[:, 1], x[:, 0]], axis=1)

In [ ]:
# Sanity check: constant velocity
x_test = np.array([[1.0, 2.0], [3.0, 4.0]])
a_test = np.array([1.0, 0.0])
v_const = constant_velocity(x_test, 0.0, a_test)
print(f'constant_velocity result:\n{v_const}')
print(f'Expected: each row = {a_test}')

# Sanity check: rotating velocity
x_rot = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]])
v_rot = rotating_velocity_2d(x_rot, 0.0)
print(f'\nrotating_velocity_2d at cardinal points:')
for xi, vi in zip(x_rot, v_rot):
    print(f'  x={xi}  ->  v={vi}')
print('Expected: [1,0]->[0,1], [0,1]->[-1,0], [-1,0]->[0,-1]  (90-degree rotation)')

In [ ]:
# Visualise rotating velocity field as a quiver plot
g = np.linspace(-2, 2, 12)
G1, G2 = np.meshgrid(g, g)
x_grid = np.stack([G1.ravel(), G2.ravel()], axis=1)
v_grid = rotating_velocity_2d(x_grid, 0.0)

plt.figure()
plt.quiver(x_grid[:, 0], x_grid[:, 1],
           v_grid[:, 0], v_grid[:, 1],
           alpha=0.8, color='tab:blue')
plt.title('Rotating velocity field: v(x, t) = [-x2, x1]')
plt.xlabel('x1')
plt.ylabel('x2')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

---
## Part 2: Euler ODE Solver

The ODE $\frac{dx_t}{dt} = v(x_t, t)$ with initial condition $x_0$ can be solved numerically.

**Euler method** (simplest first-order integrator):
$$x_{t + \Delta t} = x_t + \Delta t \cdot v(x_t, t)$$

The Euler method is easy to implement but can accumulate errors for large step sizes. More accurate methods (Runge–Kutta 4, adaptive solvers) use multiple function evaluations per step.

In [ ]:
def euler_solve(v_fn, x0, t_start=0.0, t_end=1.0, n_steps=100):
    """Solve dx/dt = v_fn(x, t) using the Euler method.

    Args:
        v_fn    : callable, v_fn(x, t) -> np.ndarray, same shape as x
        x0      : np.ndarray (D,) or (N, D), initial condition at t_start
        t_start : float
        t_end   : float
        n_steps : int

    Returns:
        x_final    : np.ndarray, same shape as x0
        trajectory : np.ndarray (n_steps+1, *x0.shape), full trajectory
    """
    # TODO:
    # dt = (t_end - t_start) / n_steps
    # x = x0.copy().astype(float)
    # trajectory = [x.copy()]
    # for i in range(n_steps):
    #     t = t_start + i * dt
    #     x = x + dt * v_fn(x, t)
    #     trajectory.append(x.copy())
    # return x, np.array(trajectory)
    dt = (t_end - t_start) / n_steps
    x = x0.copy().astype(float)
    trajectory = [x.copy()]
    for i in range(n_steps):
        t = t_start + i * dt
        x = x + dt * v_fn(x, t)
        trajectory.append(x.copy())
    return x, np.array(trajectory)

In [ ]:
# Test 1: constant velocity a=[1, 0] from x0=[0, 0] should reach [1, 0]
a_vec = np.array([1.0, 0.0])
x0_const = np.array([0.0, 0.0])
v_const_fn = lambda x, t: constant_velocity(x, t, a_vec)

x_final_const, traj_const = euler_solve(v_const_fn, x0_const, n_steps=100)
print(f'Constant velocity test:')
print(f'  x0 = {x0_const}')
print(f'  x_final = {x_final_const}  (expected [1.0, 0.0])')
print(f'  Error = {np.abs(x_final_const - np.array([1.0, 0.0])).max():.2e}')

In [ ]:
# Test 2: rotating velocity from x0=[1, 0] should trace a unit circle
x0_rot = np.array([[1.0, 0.0]])
x_final_rot, traj_rot = euler_solve(rotating_velocity_2d, x0_rot,
                                     t_start=0.0, t_end=2 * np.pi, n_steps=500)

traj_rot_2d = traj_rot[:, 0, :]  # shape (501, 2)
radius = np.linalg.norm(traj_rot_2d, axis=1)
print(f'Rotating velocity test:')
print(f'  Trajectory radius: min={radius.min():.4f}, max={radius.max():.4f}  (expected ≈ 1.0)')
print(f'  Final point: {x_final_rot[0].round(4)}  (expected ≈ [1.0, 0.0] after full rotation)')

plt.figure()
plt.plot(traj_rot_2d[:, 0], traj_rot_2d[:, 1], lw=1.5, color='tab:blue', label='Euler trajectory')
theta = np.linspace(0, 2 * np.pi, 300)
plt.plot(np.cos(theta), np.sin(theta), '--', color='tab:orange', alpha=0.7, label='True unit circle')
plt.scatter([1.0], [0.0], color='green', zorder=5, s=80, label='Start x0=[1,0]')
plt.title('Rotating velocity field: Euler trajectory vs unit circle')
plt.xlabel('x1')
plt.ylabel('x2')
plt.axis('equal')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Part 3: Divergence of a Vector Field

The **divergence** of a vector field $v = (v_1, v_2)$ in 2D is:
$$\text{div}(v) = \frac{\partial v_1}{\partial x_1} + \frac{\partial v_2}{\partial x_2}$$

Interpretation:
- $\text{div}(v) > 0$: field **expands** (particles spread out → probability density decreases)
- $\text{div}(v) < 0$: field **compresses** (particles converge → probability density increases)
- $\text{div}(v) = 0$: **volume-preserving** (probability density unchanged along trajectories)

The divergence appears in the **continuity equation**:
$$\frac{\partial p_t}{\partial t} = -\nabla \cdot (p_t \, v_t)$$

which describes how the probability density $p_t$ changes over time.

In [ ]:
def numerical_divergence(v_fn, x, t, eps=1e-4, **kwargs):
    """Estimate divergence of v_fn at point x via central finite differences.

    div(v)(x) = sum_d (v_fn(x + eps*e_d) - v_fn(x - eps*e_d))[d] / (2*eps)

    where e_d is the unit vector in dimension d.

    Args:
        v_fn   : callable, v_fn(x, t, **kwargs) -> np.ndarray (D,)
        x      : np.ndarray (D,), single point
        t      : float
        eps    : float, finite difference step
        **kwargs: extra keyword arguments forwarded to v_fn

    Returns:
        div : float
    """
    D = len(x)
    # TODO:
    # div = 0.0
    # for d in range(D):
    #     e_d = np.zeros(D); e_d[d] = 1.0
    #     div += (v_fn(x + eps*e_d, t, **kwargs)[d] - v_fn(x - eps*e_d, t, **kwargs)[d]) / (2 * eps)
    # return div
    div = 0.0
    for d in range(D):
        e_d = np.zeros(D)
        e_d[d] = 1.0
        div += (v_fn(x + eps * e_d, t, **kwargs)[d] - v_fn(x - eps * e_d, t, **kwargs)[d]) / (2 * eps)
    return div

In [ ]:
# Sanity check 1: constant velocity — divergence should be 0
x_pt = np.array([1.0, 2.0])
div_const = numerical_divergence(constant_velocity, x_pt, 0.0, a=np.array([1.0, 0.0]))
print(f'div(constant velocity) at {x_pt}: {div_const:.6f}  (expected 0.0)')

# Sanity check 2: rotating velocity — divergence should be 0 (volume-preserving)
x_rot_pt = np.array([1.0, 0.0])
div_rot = numerical_divergence(
    lambda x, t: rotating_velocity_2d(x[np.newaxis], t)[0],
    x_rot_pt, 0.0
)
print(f'div(rotating velocity) at {x_rot_pt}: {div_rot:.6f}  (expected 0.0, rotation is volume-preserving)')

# Sanity check 3: linear expanding field v(x) = x — divergence should be D = 2
linear_expand = lambda x, t: x  # v(x, t) = x
div_expand = numerical_divergence(linear_expand, x_pt, 0.0)
print(f'div(v(x)=x) at {x_pt}: {div_expand:.6f}  (expected 2.0 in 2D)')

---
## Part 4: Log-Likelihood via the Instantaneous Change of Variables

For a CNF, the log-likelihood at time $t=1$ is given by the **instantaneous change of variables formula**:

$$\log p_1(x_1) = \log p_0(x_0) - \int_0^1 \text{div}(v_t(x_t))\, dt$$

where $x_0$ and the trajectory are obtained by solving the ODE **backwards** from $x_1$.

Intuitively: if the flow expands volume ($\text{div} > 0$), probability mass spreads out and density decreases. The integral of divergence tracks exactly how much the log-probability changes along the trajectory.

We integrate the log-probability change **alongside** the ODE (the augmented ODE approach):
$$\frac{d}{dt}\begin{pmatrix} x_t \\ \Delta \end{pmatrix} = \begin{pmatrix} v(x_t, t) \\ \text{div}(v)(x_t, t) \end{pmatrix}$$

In [ ]:
def cnf_log_likelihood(x1, v_fn, base_log_prob_fn, n_steps=100):
    """Compute log p_1(x_1) for a CNF by solving the augmented ODE backwards.

    Solve the augmented system from t=1 to t=0:
        dx/dt  = -v(x, t)          (run ODE in reverse)
        dΔ/dt  = +div(v)(x, t)     (accumulate log-det change; sign convention below)

    Then: log p_1(x_1) = log p_0(x_0) + Δ

    where Δ = ∫_0^1 div(v_t) dt   (positive contribution = expanding flow → lower density)

    Wait — let's be careful with signs:
        log p_1(x_1) = log p_0(x_0) - ∫_0^1 div(v_t) dt
    Running backwards, we accumulate div with a positive sign, then subtract:
        delta_log_prob += dt * div   (integrating |div| going backwards)
        log_prob = base_log_prob_fn(x0) + delta_log_prob

    Args:
        x1               : np.ndarray (D,), point at t=1
        v_fn             : callable, v_fn(x, t) -> np.ndarray (D,)
        base_log_prob_fn : callable, log_p0(x) -> float
        n_steps          : int

    Returns:
        log_prob : float, log p_1(x_1)
    """
    # TODO:
    # 1. Run Euler backwards (t: 1 → 0) to get x_0 and accumulate div:
    #    x = x1.copy().astype(float)
    #    delta_log_prob = 0.0
    #    dt = 1.0 / n_steps
    #    for i in range(n_steps):
    #        t = 1.0 - i * dt           (current time, counting down)
    #        div = numerical_divergence(v_fn, x, t)
    #        x = x - dt * v_fn(x, t)   (reverse Euler step)
    #        delta_log_prob += dt * div (integrate divergence)
    # 2. log_prob = base_log_prob_fn(x) + delta_log_prob
    # 3. return log_prob
    x = x1.copy().astype(float)
    delta_log_prob = 0.0
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = 1.0 - i * dt
        div = numerical_divergence(v_fn, x, t)
        x = x - dt * v_fn(x, t)
        delta_log_prob += dt * div
    log_prob = base_log_prob_fn(x) + delta_log_prob
    return log_prob

In [ ]:
# Verification: constant velocity a=[1, 0] with base N(0, I)
# The flow maps N(0, I) -> N([1, 0], I) since x1 = x0 + [1, 0]
# So log p_1(x_1) should match log N(x_1 | [1, 0], I)

def base_log_prob_normal(x):
    """Log probability of standard normal N(0, I)."""
    return np.sum(norm.logpdf(x, 0, 1))

a_shift = np.array([1.0, 0.0])
v_fn_const = lambda x, t: constant_velocity(x, t, a_shift)

# Test a few points
test_points = [np.array([1.0, 0.0]), np.array([2.0, 1.0]), np.array([0.5, -0.5])]

print('Verification: CNF log-likelihood vs analytic N([1,0], I)')
print(f'{"x1":20s}  {"CNF log-prob":>14s}  {"Analytic":>14s}  {"Error":>10s}')
for x1 in test_points:
    cnf_lp = cnf_log_likelihood(x1, v_fn_const, base_log_prob_normal, n_steps=200)
    analytic_lp = norm.logpdf(x1[0], 1.0, 1.0) + norm.logpdf(x1[1], 0.0, 1.0)
    print(f'{str(x1):20s}  {cnf_lp:14.6f}  {analytic_lp:14.6f}  {abs(cnf_lp - analytic_lp):10.2e}')

---
## Part 5: Flow Matching — Conditional Paths

**Flow matching** (Lipman et al. 2022) trains the velocity field **directly** without solving ODEs during training.

**Key idea**: for each data point $x_1 \sim p_{\text{data}}$ and noise sample $x_0 \sim \mathcal{N}(0, I)$, define a **conditional linear path**:
$$x_t = (1 - t) \cdot x_0 + t \cdot x_1$$

The **conditional velocity** along this path is constant:
$$u_t(x_t \mid x_0, x_1) = x_1 - x_0$$

This is simply the direction from noise to data — the velocity doesn't depend on $t$ or $x_t$ when conditioned on the pair $(x_0, x_1)$.

The **marginal velocity** (averaged over all pairs) is what the network learns to approximate.

In [ ]:
def linear_path(x0, x1, t):
    """Linear interpolation between x0 (noise) and x1 (data) at time t.

    x_t = (1 - t) * x0 + t * x1

    Args:
        x0 : np.ndarray (N, D) or (D,), noise samples
        x1 : np.ndarray (N, D) or (D,), data samples
        t  : float, in [0, 1]

    Returns:
        xt : np.ndarray, same shape as x0/x1
    """
    # TODO: return (1 - t) * x0 + t * x1
    return (1 - t) * x0 + t * x1


def conditional_velocity(x0, x1):
    """Conditional velocity u_t(x_t | x0, x1) = x1 - x0 (constant along path).

    Args:
        x0 : np.ndarray (N, D)
        x1 : np.ndarray (N, D)

    Returns:
        velocity : np.ndarray (N, D)
    """
    # TODO: return x1 - x0
    return x1 - x0

In [ ]:
# Sanity checks
x0_demo = np.array([[0.0, 0.0], [1.0, 1.0]])
x1_demo = np.array([[2.0, 2.0], [3.0, -1.0]])

print('linear_path sanity checks:')
print(f'  t=0.0: {linear_path(x0_demo, x1_demo, 0.0)}  (expected = x0)')
print(f'  t=1.0: {linear_path(x0_demo, x1_demo, 1.0)}  (expected = x1)')
print(f'  t=0.5: {linear_path(x0_demo, x1_demo, 0.5)}  (expected midpoints)')

vel_demo = conditional_velocity(x0_demo, x1_demo)
print(f'\nconditional_velocity: {vel_demo}  (expected x1 - x0)')

In [ ]:
# Visualise linear paths and velocity arrows for 5 pairs (x0, x1)
np.random.seed(7)
x0_pairs = np.random.randn(5, 2)
x1_pairs = np.random.randn(5, 2) * 0.5 + np.array([2.0, 1.0])

t_vals = np.linspace(0, 1, 50)
colors = plt.cm.tab10(np.arange(5))

plt.figure()
for i in range(5):
    path = np.array([linear_path(x0_pairs[i], x1_pairs[i], t) for t in t_vals])
    plt.plot(path[:, 0], path[:, 1], color=colors[i], alpha=0.7, lw=1.5)
    plt.scatter(*x0_pairs[i], color=colors[i], marker='o', s=80, zorder=5)
    plt.scatter(*x1_pairs[i], color=colors[i], marker='*', s=120, zorder=5)

    # Velocity arrow at t=0.5
    xt_half = linear_path(x0_pairs[i], x1_pairs[i], 0.5)
    vel = conditional_velocity(x0_pairs[[i]], x1_pairs[[i]])[0]
    plt.annotate('', xy=xt_half + 0.15 * vel, xytext=xt_half,
                 arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5))

plt.scatter([], [], marker='o', color='gray', label='x0 (noise)', s=80)
plt.scatter([], [], marker='*', color='gray', label='x1 (data)', s=120)
plt.title('Flow matching: linear paths and conditional velocity arrows at t=0.5')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Part 6: Flow Matching Loss and Velocity Network (PyTorch)

Flow matching trains $v_\theta$ to match the conditional velocity at random times:

$$\mathcal{L}_{\text{FM}}(\theta) = \mathbb{E}_{t \sim U[0,1],\, x_1 \sim p_{\text{data}},\, x_0 \sim \mathcal{N}(0,I)} \left[ \|v_\theta(x_t, t) - (x_1 - x_0)\|^2 \right]$$

where $x_t = (1-t)x_0 + t x_1$.

This is a simple **regression** problem: the network predicts the straight-line direction from noise to data, evaluated at a random interpolation time. No ODE solving is needed during training!

In [ ]:
import torch
import torch.nn as nn

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
class VelocityNet(nn.Module):
    """Velocity network v_θ(x, t) for 2D flow matching.

    Architecture:
        Concatenate x (2-dim) and t (1-dim)
        -> Linear(3, H) -> SiLU -> Linear(H, H) -> SiLU -> Linear(H, 2)

    SiLU (sigmoid linear unit) is used because it is smooth and works well
    for velocity-field regression.
    """

    def __init__(self, hidden_dim=128):
        super().__init__()
        # TODO:
        # self.net = nn.Sequential(
        #     nn.Linear(3, hidden_dim),
        #     nn.SiLU(),
        #     nn.Linear(hidden_dim, hidden_dim),
        #     nn.SiLU(),
        #     nn.Linear(hidden_dim, 2),
        # )
        raise NotImplementedError

    def forward(self, x, t):
        """
        Args:
            x : Tensor (B, 2)
            t : Tensor (B, 1) or (B,), time in [0, 1]

        Returns:
            v : Tensor (B, 2), predicted velocity
        """
        # TODO:
        # t_in = t.view(-1, 1).float()
        # x_in = torch.cat([x, t_in], dim=1)
        # return self.net(x_in)
        raise NotImplementedError

In [ ]:
def flow_matching_loss(model, x1, x0, t):
    """Flow matching loss: || v_theta(x_t, t) - (x1 - x0) ||^2.

    Args:
        model : VelocityNet
        x1    : Tensor (B, 2), data samples
        x0    : Tensor (B, 2), noise samples from N(0, I)
        t     : Tensor (B,), times in [0, 1]

    Returns:
        loss : scalar Tensor
    """
    # TODO:
    # 1. x_t = (1 - t.view(-1, 1)) * x0 + t.view(-1, 1) * x1
    # 2. target = x1 - x0
    # 3. v_hat = model(x_t, t)
    # 4. return ((v_hat - target)**2).sum(dim=1).mean()
    raise NotImplementedError

In [ ]:
def train_flow_matching(X_data, n_epochs=400, lr=1e-3, batch_size=256):
    """Train a flow matching velocity network on 2D data.

    At each step:
      - Sample a batch of data points x1
      - Sample noise x0 ~ N(0, I)
      - Sample random times t ~ U[0, 1]
      - Regress v_theta(x_t, t) onto the conditional velocity (x1 - x0)

    Args:
        X_data   : np.ndarray (N, 2)
        n_epochs : int
        lr       : float
        batch_size : int

    Returns:
        model        : trained VelocityNet
        loss_history : list of float
    """
    X_t = torch.tensor(X_data, dtype=torch.float32).to(device)
    model = VelocityNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_t),
        batch_size=batch_size, shuffle=True
    )
    loss_history = []
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for (x1_batch,) in loader:
            # TODO:
            # 1. opt.zero_grad()
            # 2. x0 = torch.randn_like(x1_batch)
            # 3. t = torch.rand(len(x1_batch), device=device)
            # 4. loss = flow_matching_loss(model, x1_batch, x0, t)
            # 5. loss.backward(); opt.step()
            # 6. epoch_loss += loss.item()
            raise NotImplementedError
        loss_history.append(epoch_loss / len(loader))
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch + 1}  loss={loss_history[-1]:.4f}')
    return model, loss_history

In [ ]:
# Sanity check: VelocityNet instantiation and output shapes
torch.manual_seed(0)
try:
    _vnet = VelocityNet(hidden_dim=32).to(device)
    _x_in = torch.randn(8, 2).to(device)
    _t_in = torch.rand(8).to(device)
    _v_out = _vnet(_x_in, _t_in)
    print(f'VelocityNet output shape: {_v_out.shape}  (expected (8, 2))')
    n_params = sum(p.numel() for p in _vnet.parameters())
    print(f'Number of parameters: {n_params}')
    print('VelocityNet instantiation OK.')
except NotImplementedError:
    print('NotImplementedError: implement VelocityNet first.')

---
## Part 7: Sampling and Comparison

Once trained, we generate samples from a flow matching model by solving the learned ODE:
$$x_{t + \Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t), \quad x_0 \sim \mathcal{N}(0, I)$$

We use the two moons dataset as our target distribution — a standard 2D benchmark.

In [ ]:
from sklearn.datasets import make_moons

np.random.seed(42)
X_moons, _ = make_moons(n_samples=2000, noise=0.05)
X_moons = (X_moons - X_moons.mean(axis=0)) / X_moons.std(axis=0)
X_moons = X_moons.astype(np.float32)

plt.figure()
plt.scatter(X_moons[:, 0], X_moons[:, 1], s=8, alpha=0.5, color='tab:blue')
plt.title('Training data: two moons')
plt.xlabel('x1')
plt.ylabel('x2')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

print(f'Dataset shape: {X_moons.shape}')
print(f'Mean: {X_moons.mean(axis=0).round(3)},  Std: {X_moons.std(axis=0).round(3)}')

In [ ]:
# Train VelocityNet on two moons — complete all TODO stubs first!
torch.manual_seed(42)
np.random.seed(42)
try:
    model_fm, loss_history_fm = train_flow_matching(
        X_moons, n_epochs=400, lr=1e-3, batch_size=256
    )
    print('Training complete.')
except NotImplementedError:
    print('NotImplementedError: complete VelocityNet, flow_matching_loss, and train_flow_matching first.')

In [ ]:
def sample_flow_matching(model, n_samples, n_steps=100):
    """Generate samples by solving dx/dt = v_theta(x, t) from t=0 to t=1 using Euler.

    Args:
        model     : trained VelocityNet
        n_samples : int
        n_steps   : int, number of Euler integration steps

    Returns:
        samples : np.ndarray (n_samples, 2)
    """
    model.eval()
    with torch.no_grad():
        # TODO:
        # 1. x = torch.randn(n_samples, 2).to(device)   (start from N(0, I))
        # 2. dt = 1.0 / n_steps
        # 3. for i in range(n_steps):
        #      t = torch.full((n_samples,), i * dt, device=device)
        #      x = x + dt * model(x, t)
        # 4. return x.cpu().numpy()
        raise NotImplementedError

In [ ]:
# Generate samples and compare to training data
try:
    samples_fm = sample_flow_matching(model_fm, n_samples=2000, n_steps=100)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(X_moons[:, 0], X_moons[:, 1], s=8, alpha=0.4, color='tab:blue')
    axes[0].set_title('Training data (two moons)')
    axes[0].set_xlabel('x1')
    axes[0].set_ylabel('x2')
    axes[0].set_aspect('equal')
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(samples_fm[:, 0], samples_fm[:, 1], s=8, alpha=0.5, color='tab:orange')
    axes[1].set_title('Flow matching generated samples')
    axes[1].set_xlabel('x1')
    axes[1].set_ylabel('x2')
    axes[1].set_aspect('equal')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Training data vs. Flow Matching samples')
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print('NotImplementedError: complete sample_flow_matching first.')
except NameError:
    print('NameError: train the model first (run the training cell).')

In [ ]:
# Visualise ODE trajectories: noise -> data for a few samples
def sample_with_trajectories(model, n_samples, n_steps=50):
    """Generate samples and record full ODE trajectories.

    Args:
        model     : trained VelocityNet
        n_samples : int
        n_steps   : int

    Returns:
        trajectories : np.ndarray (n_steps+1, n_samples, 2)
    """
    model.eval()
    with torch.no_grad():
        # TODO:
        # x = torch.randn(n_samples, 2).to(device)
        # dt = 1.0 / n_steps
        # traj = [x.cpu().numpy().copy()]
        # for i in range(n_steps):
        #     t = torch.full((n_samples,), i * dt, device=device)
        #     x = x + dt * model(x, t)
        #     traj.append(x.cpu().numpy().copy())
        # return np.array(traj)
        raise NotImplementedError

try:
    torch.manual_seed(3)
    trajs = sample_with_trajectories(model_fm, n_samples=10, n_steps=50)
    # trajs: (51, 10, 2)

    plt.figure()
    colors_traj = plt.cm.tab10(np.arange(10))
    for i in range(10):
        plt.plot(trajs[:, i, 0], trajs[:, i, 1], lw=1.2, alpha=0.7, color=colors_traj[i])
        plt.scatter(*trajs[0, i], marker='o', s=60, color=colors_traj[i], zorder=5)
        plt.scatter(*trajs[-1, i], marker='*', s=100, color=colors_traj[i], zorder=5)

    plt.scatter([], [], marker='o', color='gray', s=60, label='t=0 (noise)')
    plt.scatter([], [], marker='*', color='gray', s=100, label='t=1 (generated)')
    plt.title('Flow matching ODE trajectories: noise → data')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
except NotImplementedError:
    print('NotImplementedError: complete sample_with_trajectories first.')
except NameError:
    print('NameError: train the model first.')

In [ ]:
# Plot training loss curve
try:
    plt.figure()
    plt.plot(loss_history_fm, color='tab:blue', lw=1.5)
    plt.xlabel('Epoch')
    plt.ylabel('Flow matching loss')
    plt.title('Flow matching training loss')
    plt.grid(True, alpha=0.3)
    plt.show()
except NameError:
    print('NameError: train the model first.')

---
## Part 8: CNF vs Flow Matching Comparison

| Aspect | Discrete NF (Assignment 8) | Flow Matching (this assignment) |
|--------|---------------------------|--------------------------------|
| Training signal | Exact log-likelihood | Regression on conditional velocity |
| Requires ODE during training? | No | No |
| Requires ODE during sampling? | No (single forward pass) | Yes (Euler integration) |
| Architectural constraint | Strictly invertible layers | Unconstrained MLP |
| Log-likelihood at test time | Exact | Approximate (via ODE integration) |

**Why flow matching is attractive**:
- The velocity network $v_\theta$ has **no invertibility constraints** — it's just an MLP
- Training is a simple **supervised regression** — no ODE solving, no log-det computation
- Sampling quality can be improved by using **more Euler steps** or better ODE solvers
- Scales naturally to **high dimensions** (images, audio, molecules)

**Reflection question 1**: What does divergence = 0 mean geometrically for the flow? What happens to the probability density along a volume-preserving trajectory?

**Your answer**: TODO

**Reflection question 2**: Flow matching trains on conditional velocities (given both $x_0$ and $x_1$). Why does minimising the conditional objective also minimise the true (marginal) flow matching objective — i.e., why does conditioning on individual pairs still produce a valid marginal velocity field?

**Your answer**: TODO

---
## Summary

In this notebook you:

- Defined and visualised **velocity fields** in 2D (constant and rotating)
- Implemented an **Euler ODE solver** and traced particle trajectories
- Computed the **divergence** of vector fields numerically via finite differences
- Derived and implemented the **instantaneous change-of-variables formula** for CNF log-likelihood
- Implemented **flow matching**: linear conditional paths and constant conditional velocity
- Built and trained a **VelocityNet** using the flow matching regression loss
- Generated new samples by **Euler-integrating** the learned velocity field
- Compared flow matching to discrete normalising flows

| Component | Role |
|---|---|
| `constant_velocity` | Simplest velocity field: uniform translation |
| `rotating_velocity_2d` | Volume-preserving rotation field |
| `euler_solve` | Numerical ODE solver: $x_{t+\Delta t} = x_t + \Delta t \cdot v(x_t, t)$ |
| `numerical_divergence` | Finite-difference estimate of $\text{div}(v)$ |
| `cnf_log_likelihood` | Log-likelihood via augmented reverse ODE |
| `linear_path` | Conditional interpolation: $x_t = (1-t)x_0 + t x_1$ |
| `conditional_velocity` | Constant conditional velocity: $u = x_1 - x_0$ |
| `VelocityNet` | PyTorch MLP predicting velocity from $(x, t)$ |
| `flow_matching_loss` | Regression loss: $\|v_\theta(x_t, t) - (x_1 - x_0)\|^2$ |
| `train_flow_matching` | Full training loop for flow matching |
| `sample_flow_matching` | ODE-based sampling: $x_0 \sim \mathcal{N}(0,I)$, integrate to $t=1$ |

**Next**: Assignment 10 returns to score-based models and covers multi-scale noise, annealed Langevin dynamics, and the connection between score-based and diffusion models.